# Lab 18 — LoRA and QLoRA: Measure the Resource Trade-off

Use a real open-weight model and treat LoRA rank and quantization as experimental variables.

**Free-first anchor:** Qwen3-0.6B. The notebook is intentionally configurable so the same experiment can later be moved to a larger model on an H100.


In [ ]:
!pip -q install -U transformers datasets trl accelerate peft bitsandbytes
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/2**30,2))


## 1. LoRA parameter accounting
For a linear layer, full parameters are d_in × d_out and LoRA parameters are r × (d_in + d_out).

In [ ]:
def lora_params(in_dim,out_dim,r): return r*(in_dim+out_dim)
full=4096*4096
for r in [4,8,16,32,64]:
    p=lora_params(4096,4096,r)
    print(f'r={r:>2}  adapter={p:,}  ratio={p/full:.6f}')


## 2. Load a real model and attach LoRA
This is the actual PEFT path. The target modules are intentionally configurable; students should inspect the architecture before choosing them.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
MODEL_ID='Qwen/Qwen3-0.6B'
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None,
)
config=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,target_modules='all-linear',task_type='CAUSAL_LM')
model=get_peft_model(model,config)
model.print_trainable_parameters()


## 3. QLoRA configuration
For a quantized base, Hugging Face recommends a 4-bit BitsAndBytesConfig and preparation for k-bit training before attaching the adapter. Measure the actual memory rather than assuming a fixed saving.

In [ ]:
from transformers import BitsAndBytesConfig
from peft import prepare_model_for_kbit_training
if torch.cuda.is_available():
    bnb_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    )
    qmodel=AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto')
    qmodel=prepare_model_for_kbit_training(qmodel)
    qmodel=get_peft_model(qmodel,config)
    qmodel.print_trainable_parameters()
    print('QLoRA allocated GiB:', round(torch.cuda.memory_allocated()/2**30,2))


## 4. Controlled experiment matrix
Run the same dataset and training budget for:

A. LoRA rank 8
B. LoRA rank 16
C. LoRA rank 32
D. QLoRA rank 16

Report:
- trainable parameters;
- peak memory;
- tokens/sec;
- wall time;
- target score;
- retained capability;
- adapter size.


## 5. Failure diagnosis
**OOM:** inspect sequence length, activations, batch, optimizer, and quantization.

**Quality drop:** compare LoRA and QLoRA on the same evaluation set.

**Unexpectedly slow QLoRA:** profile kernels, dequantization, data loading, and checkpointing.

**Rank increase does not help:** the bottleneck may be data coverage rather than adapter capacity.


## Primary sources
- PEFT quantization guide: https://huggingface.co/docs/peft/developer_guides/quantization
- LoRA: https://arxiv.org/abs/2106.09685
- QLoRA: https://arxiv.org/abs/2305.14314
